# Example 10 — Dalitz + discriminating variables

This notebook adds reconstructed B mass and a BDT-like observable to the Dalitz fit using a factorized PDF.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (BackgroundCategory, DecayChannel, DecayModel, Exponential1D, FactorizedDensity, Gaussian1D, Histogram1D, Minimizer, MultiBackgroundNLL, NonResonant, Parameter, PhaseSpaceSample, RealImag, Resonance, enable_x64, weighted_resample)
from dalitzplotfitter.background import FunctionalBackground
enable_x64()
channel=DecayChannel('B+',('K+','pi+','pi-'))
model=DecayModel(channel,[Resonance('Kstar892',(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),Resonance('rho770',(1,2),RealImag(0.65,0.10),mass=0.7753,width=0.1491,spin=1),NonResonant(RealImag(-0.5,0.1))],normalization_method='square-dalitz',normalization_resolution=250,normalization_pair=(0,2))


## 1. Generate Dalitz signal/background plus mass and BDT observables


In [ ]:
rng=np.random.default_rng(10001)
pool=model.generate_phase_space(120000,seed=10002); norm=model.normalization_sample
bkg_shape=FunctionalBackground(lambda d:0.4+0.8*(d['s13']-jnp.min(norm.s13))/(jnp.max(norm.s13)-jnp.min(norm.s13)))
N=25000; FS_TRUE=0.70; ns=int(round(N*FS_TRUE)); nb=N-ns
sig=weighted_resample(jax.random.key(10003),pool,pool.weights*model.intensity(pool.as_dict()),ns,replace=True)
bkg=weighted_resample(jax.random.key(10004),pool,pool.weights*bkg_shape(pool.as_dict()),nb,replace=True)
def merge(a,b):
    def c(name):
        x,y=getattr(a,name),getattr(b,name); return None if x is None else jnp.concatenate((x,y))
    return PhaseSpaceSample(s12=c('s12'),s13=c('s13'),s23=c('s23'),weights=jnp.ones(a.size+b.size),p1=c('p1'),p2=c('p2'),p3=c('p3'))
data=merge(sig,bkg)
mass=np.concatenate((rng.normal(5.279,0.014,ns),5.20+rng.exponential(0.055,nb)))
mass=np.clip(mass,5.20,5.35)
bdt=np.concatenate((rng.beta(5.0,1.8,ns),rng.beta(1.4,4.0,nb)))
perm=rng.permutation(N)
data=data.take(jnp.asarray(perm)); mass=jnp.asarray(mass[perm]); bdt=jnp.asarray(bdt[perm])
print('events:',N,'signal:',ns,'background:',nb)


## 2. Factorized component PDFs

For each component we assume $P(DP,m_B,BDT)=P(DP)P(m_B)P(BDT)$.


In [ ]:
d=data.as_dict(); signal_dp=model.pdf()
mass_mean=Parameter('mass_mean',5.270,bounds=(5.24,5.31),step=0.001)
sig_mass=Gaussian1D(mass_mean,0.014,5.20,5.35)
bkg_mass=Exponential1D(-8.0,5.20,5.35)
edges=jnp.linspace(0,1,11)
sig_bdt=Histogram1D(edges,jnp.array([0.05,0.08,0.12,0.20,0.35,0.60,0.95,1.35,1.75,2.10]))
bkg_bdt=Histogram1D(edges,jnp.array([2.20,1.80,1.35,0.95,0.65,0.40,0.25,0.15,0.08,0.04]))
signal_full=FactorizedDensity(lambda v:signal_dp(d,v),{'mass':mass,'bdt':bdt},{'mass':sig_mass,'bdt':sig_bdt})
bkg_norm=jnp.mean(norm.weights*bkg_shape(norm.as_dict()))
bkg_dp=bkg_shape(d)/bkg_norm
bkg_full=bkg_dp*bkg_mass(mass)*bkg_bdt(bdt)
f_sig=Parameter('signal_fraction',0.60,bounds=(0.01,0.99),step=0.01)
category=BackgroundCategory('combinatorial',bkg_full,1.0)
nll=MultiBackgroundNLL(signal_density=signal_full,backgrounds=(category,),signal_fraction=f_sig)
pars=(f_sig,mass_mean)
start={'signal_fraction':0.60,'mass_mean':5.270}
result=Minimizer(nll,pars,verbose=1).fit(start_values=start,simplex=True,ncall=20000)
fit={p.name:float(result.values[p.name]) for p in pars}
print('valid:',result.valid)
print('signal fraction true/start/fit:',FS_TRUE,start['signal_fraction'],fit['signal_fraction'])
print('mass mean true/start/fit:',5.279,start['mass_mean'],fit['mass_mean'])


## 3. Mass and BDT projections


In [ ]:
x=np.linspace(5.20,5.35,500); fs=fit['signal_fraction']
pm=fs*np.asarray(sig_mass(jnp.asarray(x),fit))+(1-fs)*np.asarray(bkg_mass(jnp.asarray(x)))
plt.figure(figsize=(7,5)); counts,bins,_=plt.hist(np.asarray(mass),bins=60,range=(5.20,5.35),density=True,alpha=0.35,label='toy data'); plt.plot(x,pm,label='fitted marginal PDF'); plt.xlabel(r'$m_B$ [GeV]'); plt.ylabel('density'); plt.legend(); plt.show()
centers=0.5*(np.asarray(edges[:-1])+np.asarray(edges[1:])); widths=np.diff(np.asarray(edges)); pb=fs*np.asarray(sig_bdt(centers,fit))+(1-fs)*np.asarray(bkg_bdt(centers,fit))
plt.figure(figsize=(7,5)); plt.hist(np.asarray(bdt),bins=np.asarray(edges),density=True,alpha=0.35,label='toy data'); plt.step(np.r_[np.asarray(edges[:-1]),edges[-1]],np.r_[pb,pb[-1]],where='post',label='fitted marginal PDF'); plt.xlabel('BDT'); plt.ylabel('density'); plt.legend(); plt.show()


## 4. Dalitz projection after the joint fit


In [ ]:
plt.figure(figsize=(7,5.5)); h=plt.hist2d(np.asarray(data.s13),np.asarray(data.s23),bins=70); plt.colorbar(h[3]); plt.xlabel(r'$s_{13}$ [GeV$^2$]'); plt.ylabel(r'$s_{23}$ [GeV$^2$]'); plt.title('Toy data used in the joint fit'); plt.show()
